In [2]:
import gymnasium as gym
import numpy as np

In [3]:
# -------------------------------------------------
# Create FrozenLake Environment
# -------------------------------------------------

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
env = env.unwrapped

n_states = env.observation_space.n
n_actions = env.action_space.n

gamma = 0.99
theta = 1e-8

In [10]:
# -------------------------------------------------
# Policy Evaluation
# -------------------------------------------------

def policy_evaluation(env, policy, gamma, theta):

    V = np.zeros(env.observation_space.n)

    while True:

        delta = 0

        for state in range(env.observation_space.n):

            v = 0

            for action, action_prob in enumerate(policy[state]):

                for prob, next_state, reward, done in env.P[state][action]:

                    v += action_prob * prob * (
                        reward + gamma * V[next_state] * (not done)
                    )

            delta = max(delta, abs(v - V[state]))
            V[state] = v

        if delta < theta:
            break

    return V

In [11]:
# -------------------------------------------------
# Policy Improvement
# -------------------------------------------------

def policy_improvement(env, V, gamma=0.99):

    n_states = env.observation_space.n
    n_actions = env.action_space.n

    policy = np.zeros((n_states, n_actions))

    for state in range(n_states):

        action_values = np.zeros(n_actions)

        for action in range(n_actions):

            for prob, next_state, reward, done in env.P[state][action]:

                action_values[action] += prob * (
                    reward + gamma * V[next_state] * (not done)
                )

        best_action = np.argmax(action_values)

        policy[state][best_action] = 1.0

    return policy

In [12]:
# -------------------------------------------------
# Policy Iteration
# -------------------------------------------------

def policy_iteration(env, gamma=0.99, theta=1e-8):

    n_states = env.observation_space.n
    n_actions = env.action_space.n

    policy = np.ones((n_states, n_actions)) / n_actions

    iterations = 0

    while True:

        iterations += 1

        V = policy_evaluation(env, policy, gamma, theta)

        new_policy = policy_improvement(env, V, gamma)

        if np.array_equal(policy, new_policy):
            break

        policy = new_policy

    print("Total policy iterations:", iterations)

    return policy, V

In [13]:
# -------------------------------------------------
# Display Functions
# -------------------------------------------------

def print_value_function(V):
    print("\nOptimal State-Value Function:")
    print(np.round(V.reshape(4, 4), 4))


def print_policy(policy):
    action_symbols = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }

    best_actions = np.argmax(policy, axis=1)
    policy_grid = np.array(
        [action_symbols[action] for action in best_actions]
    ).reshape(4, 4)


    print("\nOptimal Policy:")
    print(policy_grid)



In [14]:
# -------------------------------------------------
# Run Policy Iteration
# -------------------------------------------------

optimal_policy, optimal_value_function = policy_iteration(
    env,
    gamma=gamma,
    theta=theta
)

print("Name: K MADHAVA REDDY")
print("Register Number: 212223240064")
print_value_function(optimal_value_function)
print_policy(optimal_policy)

env.close()

Total policy iterations: 3
Name: K MADHAVA REDDY
Register Number: 212223240064

Optimal State-Value Function:
[[0.542  0.4988 0.4707 0.4569]
 [0.5585 0.     0.3583 0.    ]
 [0.5918 0.6431 0.6152 0.    ]
 [0.     0.7417 0.8628 0.    ]]

Optimal Policy:
[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]
